# 01 — Clean HUC12 Geometries

**Purpose:** Load the NHD Watershed Boundary Dataset (WBD) shapefile, filter to the
Upper Mississippi River Basin (UMRB, HUC region 07), detect and merge any duplicate
HUC12 polygon records, and save a clean shapefile for downstream use.

**Input:** `WBD_Subwatershed.shp`  
**Output:** `UMRB_HUC12_modified.shp`

**Why duplicates exist:** Some HUC12 units in the WBD are represented as multiple
disjoint polygon records (e.g., islands or boundary artefacts). These must be
dissolved with `unary_union` before any spatial operations to avoid double-counting.

In [ ]:
# =============================================================================
# USER SETTINGS — edit these paths before running
# =============================================================================

# Root directory containing your datasets/ folder
WORKING_DIR = "/path/to/your/data"

# Input: NHD WBD shapefile
WBD_SHP = f"{WORKING_DIR}/WBD_Subwatershed.shp"

# Output: deduplicated UMRB HUC12 shapefile
HUC12_CLEAN_SHP = f"{WORKING_DIR}/UMRB_HUC12_modified.shp"

# HUC region prefix for UMRB (region 07 = Upper Mississippi River Basin)
UMRB_HUC_PREFIX = ("07",)

# =============================================================================
import os
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.ops import unary_union

warnings.filterwarnings('ignore')
%matplotlib inline

os.makedirs(os.path.dirname(HUC12_CLEAN_SHP), exist_ok=True)
print(f'Input : {WBD_SHP}')
print(f'Output: {HUC12_CLEAN_SHP}')

## 1. Load WBD and filter to UMRB

In [ ]:
WBD_HUC12 = gpd.read_file(WBD_SHP)

UMRB_HUC12 = (
    WBD_HUC12[WBD_HUC12['HUC_12'].str.startswith(UMRB_HUC_PREFIX)]
    .reset_index(drop=True)
)
print(f'Total HUC12 records in UMRB : {len(UMRB_HUC12)}')
print(f'Unique HUC_12 codes         : {UMRB_HUC12["HUC_12"].nunique()}')

## 2. Identify and visualise duplicate HUC12 records

In [ ]:
duplicates = UMRB_HUC12[UMRB_HUC12.duplicated('HUC_12', keep=False)]
print(f'Duplicate records : {len(duplicates)} across {duplicates["HUC_12"].nunique()} HUC12 codes')
display(duplicates[['HUC_12', 'HU_12_NAME', 'ACRES', 'SHAPE_Area']].head(10))

In [ ]:
if not duplicates.empty:
    fig, ax = plt.subplots(figsize=(10, 8))
    duplicates.plot(ax=ax, color='lightblue', edgecolor='black')
    ax.set_title('HUC12 polygons with duplicate records', fontsize=14)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_xticks([])
    ax.set_yticks([])
    plt.tight_layout()
    plt.show()
else:
    print('No duplicates found — no merging required.')

## 3. Merge duplicate polygons with `unary_union`

For duplicated HUC12 codes, dissolve their geometries into a single MultiPolygon.
Area-additive fields (`ACRES`, `SHAPE_Leng`, `SHAPE_Area`) are summed;
all other attributes take the first occurrence.

In [ ]:
original_crs = UMRB_HUC12.crs

aggregations = {
    'OBJECTID'  : 'first',
    'HUC_8'     : 'first',
    'HUC_10'    : 'first',
    'HUC_12'    : 'first',
    'ACRES'     : 'sum',        # area-additive
    'NCONTRB_A' : 'first',
    'HU_10_GNIS': 'first',
    'HU_12_GNIS': 'first',
    'HU_10_NAME': 'first',
    'HU_10_MOD' : 'first',
    'HU_10_TYPE': 'first',
    'HU_12_DS'  : 'first',
    'HU_12_NAME': 'first',
    'HU_12_MOD' : 'first',
    'HU_12_TYPE': 'first',
    'META_ID'   : 'first',
    'STATES'    : 'first',
    'GlobalID'  : 'first',
    'SHAPE_Leng': 'sum',        # area-additive
    'SHAPE_Area': 'sum',        # area-additive
    'GAZ_ID'    : 'first',
    'geometry'  : lambda x: unary_union(x),
}

UMRB_HUC12_clean = (
    UMRB_HUC12
    .groupby('HUC_12', as_index=False)
    .agg(aggregations)
    .pipe(gpd.GeoDataFrame, geometry='geometry', crs=original_crs)
)

print(f'Records after deduplication: {len(UMRB_HUC12_clean)}')
assert UMRB_HUC12_clean['HUC_12'].nunique() == len(UMRB_HUC12_clean), \
    'Duplicate HUC_12 codes remain after aggregation!'

## 4. Spot-check a merged polygon

In [ ]:
# Visual QC: verify that a previously duplicated polygon appears as one unit
check_huc = '070900010206'
fig, ax = plt.subplots(figsize=(8, 6))
UMRB_HUC12_clean[UMRB_HUC12_clean['HUC_12'] == check_huc].plot(
    ax=ax, color='lightblue', edgecolor='black'
)
ax.set_title(f'Merged geometry for HUC12 {check_huc}', fontsize=13)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()

## 5. Save output

In [ ]:
UMRB_HUC12_clean.to_file(HUC12_CLEAN_SHP, driver='ESRI Shapefile')
print(f'Saved: {HUC12_CLEAN_SHP}')